<a href="https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!git clone https://github.com/abdulmanan2728/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 383, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 383 (delta 158), reused 105 (delta 96), pack-reused 178 (from 2)
Receiving objects: 100% (383/383), 1.96 MiB | 7.93 MiB/s, done.
Resolving deltas: 100% (213/213), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [11]:
import pandas as pd, numpy as np, json, os
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'age_tier_order', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
X, y, groups = df[feature_cols], df['is_declining'], df['client_id']

# baseline rule
median_ctr_by_tier = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_gap'] = median_ctr_by_tier - df['ctr']
df['baseline_flagged'] = (df['freshness_tier'].isin(['31-90','91-180']) & (df['ctr_gap'] > 0)).astype(int)

# honest, grouped split (the one this paper reports)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)
baseline_preds = df.loc[X_test.index, 'baseline_flagged']

print("setup complete —", df.shape[0], "rows,", len(feature_cols), "features")

setup complete — 30000 rows, 29 features


## 1. Question

*The research question and the decision it supports.*

## 1. Question

**Lane: Refresh / Content Opportunity Scoring.** Which pages should an SEO analyst
review this week, out of a portfolio too large to check by hand? A wrong call costs
either wasted analyst time (false positive) or a page's decline going unnoticed another
cycle (false negative) — bounded costs, which is why this is decision-support, not an
automated action. Full framing in `w01_research_question.ipynb` and
`w02_ml_task_framing.ipynb`.

In [12]:
base_rate = y.mean()
print(f"content items: {len(df)}, clients: {df['client_id'].nunique()}, base rate: {base_rate:.3f}")

content items: 30000, clients: 32, base rate: 0.542


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 2. Data

FlyRank's pseudonymized starter dataset — one row per content item, 32 clients,
trailing-90-day metrics. Label `is_declining` is built from the dataset's own
`trend_direction` field — an observed outcome, not a hand-labeled judgment.
`trend_pct`/`trend_direction` are excluded as features since they're the label's source.
Full contract and verification queries in `w03_data_contract.ipynb`.

In [13]:
print(f"duplicate content_id rows: {df['content_id'].duplicated().sum()}")  # grain check, should be 0
leak_corr = df[feature_cols].corrwith(df['trend_pct']).abs().max()
print(f"max feature correlation with label source (trend_pct): {leak_corr:.3f}")

duplicate content_id rows: 0
max feature correlation with label source (trend_pct): 0.097


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 3. Methodology

Random Forest, 28 features, validated with `GroupShuffleSplit` on `client_id` — no
client appears in both train and test. Baseline: a hand-written rule (stale 31-180
days AND below-median CTR vs. position peers), each signal verified independently
before use. Full method reasoning in `w05_model.ipynb`, split/leakage audit in
`w06_validation_audit.ipynb`.

In [14]:
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
print(f"client overlap between train/test: {len(train_clients & test_clients)}")  # should be 0
print(f"train rows: {len(X_train)}, test rows: {len(X_test)}")

client overlap between train/test: 0
train rows: 22885, test rows: 7115


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 4. Results (vs baseline)

Under the honest client-grouped split, the model beats the baseline by 28 points of
precision and roughly 9x the recall. A naive random split on this same model produces
an inflated 0.901 precision — reported here for contrast, not as the real number.

In [15]:
comparison = pd.DataFrame({
    'method': ['baseline_rule', 'random_forest'],
    'precision': [precision_score(y_test, baseline_preds), precision_score(y_test, model_preds)],
    'recall':    [recall_score(y_test, baseline_preds), recall_score(y_test, model_preds)],
    'f1':        [f1_score(y_test, baseline_preds), f1_score(y_test, model_preds)],
})
print(comparison)

# naive-split comparison, for contrast only
X_tr_n, X_te_n, y_tr_n, y_te_n = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
naive_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_n, y_tr_n)
naive_precision = precision_score(y_te_n, naive_model.predict(X_te_n))
print(f"\nnaive random split precision (inflated, not reported as real): {naive_precision:.3f}")
print(f"honest grouped split precision (reported): {precision_score(y_test, model_preds):.3f}")

          method  precision    recall        f1
0  baseline_rule   0.524217  0.100136  0.168152
1  random_forest   0.805353  0.900680  0.850353

naive random split precision (inflated, not reported as real): 0.901
honest grouped split precision (reported): 0.805


## 5. Limitations

*What this work cannot claim.*

## 5. Limitations

Can claim: 0.805 precision / 0.900 recall flagging a 30-day impression decline trend,
under an honest holdout — directional, decision-support. Cannot claim: causal decline
drivers, future certainty, or anything about Google's ranking algorithm. `review` +
`review_urgent` flags cover 50.5% of inventory — a ranking signal to sort within
capacity, not a fixed weekly workload. Trained on one historical snapshot; real-world
drift after training is untested. Full claim rewrite in `w06_validation_audit.ipynb`.

In [16]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.head(5))

impressions_prev_30d     0.211458
impressions_last_30d     0.157783
impressions_90d          0.080401
avg_position             0.061751
days_with_impressions    0.046095
dtype: float64


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 6. Ranked recommendations

1. Start with `review_urgent` (high risk + stale >90 days).
2. Rank `review` by `risk_score`, take only the top slice your team has capacity for.
3. Check impression volume before trusting any flag — false positives skew low-traffic.
4. Never auto-publish, auto-unpublish, or auto-deindex from the score alone.
5. Re-validate quarterly, or sooner if flagged share drifts from 50.5%.

Full playbook and no-go list in `w07_action_playbook.ipynb`.

In [17]:
df['risk_score'] = model.predict_proba(X)[:, 1]

def assign_action(row):
    if row['risk_score'] >= 0.7 and row['days_since_last_update'] > 90:
        return 'review_urgent'
    elif row['risk_score'] >= 0.7:
        return 'review'
    elif row['risk_score'] >= 0.4:
        return 'monitor'
    return 'healthy'

df['action'] = df.apply(assign_action, axis=1)
print(df['action'].value_counts())

action
healthy          12406
review            9627
review_urgent     5525
monitor           2442
Name: count, dtype: int64


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## 7. Artifacts the paper embeds

The exact tables and numbers the deployed paper's Results and Recommendations
sections show — generated here so the paper's claims trace back to this notebook.

In [18]:
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

capstone_metrics = {
    "baseline_precision": float(precision_score(y_test, baseline_preds)),
    "baseline_recall": float(recall_score(y_test, baseline_preds)),
    "model_precision": float(precision_score(y_test, model_preds)),
    "model_recall": float(recall_score(y_test, model_preds)),
    "model_f1": float(f1_score(y_test, model_preds)),
    "naive_split_precision_inflated": float(naive_precision),
    "top_features": importances.head(5).to_dict(),
    "action_counts": df['action'].value_counts().to_dict(),
    "flagged_share": float(df['action'].isin(['review','review_urgent']).mean()),
}
with open('work/outputs/capstone_metrics.json', 'w') as f:
    json.dump(capstone_metrics, f, indent=2)
print(capstone_metrics)

{'baseline_precision': 0.5242165242165242, 'baseline_recall': 0.1001360544217687, 'model_precision': 0.805352798053528, 'model_recall': 0.9006802721088435, 'model_f1': 0.8503532434168273, 'naive_split_precision_inflated': 0.900785153461813, 'top_features': {'impressions_prev_30d': 0.21145772107254474, 'impressions_last_30d': 0.1577832837035358, 'impressions_90d': 0.08040088071359983, 'avg_position': 0.06175110818913695, 'days_with_impressions': 0.046094842498280296}, 'action_counts': {'healthy': 12406, 'review': 9627, 'review_urgent': 5525, 'monitor': 2442}, 'flagged_share': 0.5050666666666667}


## ML-12: Demo, social cut, employer summary

**5-minute demo outline:**
1. (30s) The problem: too many pages, too little review time.
2. (1min) The baseline rule and its weak recall (0.100) — show it live.
3. (2min) The model, the grouped-split gap (0.901 → 0.805), and why that matters.
4. (1min) The ranked queue in action — walk through 2-3 real flagged rows.
5. (30s) Limitations, stated plainly, and the one-line takeaway.

**Social-post cut:**
> Built a content-decline scorer on FlyRank's real search data: a client-grouped
> Random Forest hits 0.805 precision vs. a hand-written rule's 0.524 — on 30K pages,
> 32 clients. The honest number came from catching my own naive-split leakage first
> (0.901 → 0.805). Paper: [https://abdulmanan2728.github.io/flyrank-ml-internship-starter/?v=2]

**3-sentence employer summary:**
Built a content-decline scoring model on FlyRank's real search performance data
(30,000 pages, 32 clients), validated with a client-grouped holdout to avoid the
leakage a naive split would hide. The model reaches 0.805 precision and 0.900 recall,
beating a hand-written baseline's 0.524/0.100 on identical test data. Delivered as a
ranked, reason-coded review queue with documented limitations and human-review rules,
not just a notebook metric.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
